# TRACE-Seg3D – toàn bộ thí nghiệm ISBI trên Kaggle (2× T4), tự chạy tiếp qua nhiều version

**Mỗi lần chỉ cần bấm _Save Version → Save & Run All (Commit)_.** Notebook tự:
1. lấy code mới nhất từ Kaggle Dataset `trace-code` của bạn;
2. tải trạng thái cũ (data đã xử lý, checkpoint, kết quả) từ 2 Kaggle Dataset private `trace-isbi-data` và `trace-isbi-runs` (tự tạo ở version đầu);
3. chạy tiếp các job theo thứ tự ưu tiên **A → B → C** trên cả 2 GPU, dừng trước 12 giờ; run đang dở sẽ train tiếp ở version sau;
4. lưu trạng thái lên lại 2 dataset trên, in bảng kết quả ở cuối log và tạo `results.zip` (bảng + audit + log, không có checkpoint) trong tab Output.

Không cần download/upload tay, không cần sửa cell nào giữa các version.

### Cài đặt 1 lần
1. **Code**: *Datasets → New Dataset* → upload `TRACE-Seg3D_ISBI.zip`, đặt tên **`trace-code`**, để Private. Khi có zip mới: mở dataset đó → *New Version* → upload zip mới (notebook tự lấy bản mới nhất).
2. **Kaggle API token**: *Settings → API Tokens → Generate New Token* (chuỗi `KGAT_...`).
3. **Trong notebook này**: *Add-ons → Secrets* → thêm 2 secret và tick bật cho notebook:
   `KAGGLE_USERNAME` = tên tài khoản Kaggle, `KAGGLE_API_TOKEN` = chuỗi `KGAT_...`.
   (Dùng *Legacy API Key* cũng được: khi đó thêm `KAGGLE_KEY` = trường `key` trong `kaggle.json` thay cho `KAGGLE_API_TOKEN`.)
4. **Settings** (cột phải): *Accelerator* = **GPU T4 x2**, *Internet* = **On** (cần xác minh số điện thoại Kaggle).
5. **Add Input**: BraTS `awsaf49/brats20-dataset-training-validation` (chỉ cần ở version đầu, sau đó data đã xử lý nằm trong `trace-isbi-data`; để nguyên cũng không sao).
   UTSW: upload thư mục `utsw_128` đã tiền xử lý trên máy bạn (hoặc bản gốc) thành dataset private rồi Add Input. Thêm UTSW lúc nào cũng được, version sau sẽ tự xử lý và chạy phần còn thiếu.


## 1. Cấu hình (không cần sửa nếu đặt tên dataset code là `trace-code`)

In [ ]:
CODE_DATASET = 'trace-code'      # tên Kaggle Dataset chứa TRACE-Seg3D_ISBI.zip

PLAN = {
    # toàn bộ ma trận; notebook tự chạy theo thứ tự ưu tiên:
    #   A = seed 0 cho 3 phương pháp (cả 2 nguồn) -> eval ngay để có bảng sơ bộ
    #   B = seed 1, 2                              -> eval lại với cả 3 seed (thêm tín hiệu ensemble)
    #   C = ablation (1 seed)
    'methods':    'baseline styleaug trace',
    'seeds':      '0 1 2',
    'ablations':  'trace_nocct trace_nostab trace_noproxy',
    'abl_source': 'utsw',          # ablation chạy trên nguồn này (chưa có UTSW thì dùng BraTS)
    # KHÔNG đổi giữa các version
    'epochs': 80,
    'extra':  '--patch-size 96 --batch-size 2 --val-every 5',   # đã đo trên T4: 112x2 bị out of memory
}

## 2. Secrets + lấy code mới nhất

In [ ]:
import glob, json, os, shutil, subprocess, sys, zipfile

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name), None
    except Exception as e:
        return None, repr(e)[:200]

found = {}
for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY', 'KAGGLE_API_TOKEN'):
    v, err = secret(k)
    if v:
        os.environ[k] = found[k] = v.strip()
if 'KAGGLE_API_TOKEN' in found and 'KAGGLE_KEY' not in found:
    # token kiểu mới (KGAT_...) cần thư viện kaggle >= 1.8
    !pip install -q -U "kaggle>=1.8.0"
SYNC = bool(found.get('KAGGLE_USERNAME') and (found.get('KAGGLE_KEY') or found.get('KAGGLE_API_TOKEN')))
print('secrets đọc được:', sorted(found), '| tự lưu trạng thái:', SYNC)
if not SYNC:
    print('   -> Add-ons -> Secrets: cần KAGGLE_USERNAME + (KAGGLE_API_TOKEN hoặc KAGGLE_KEY), và phải TICK "attached" cho notebook này.'
          ' Không có secrets thì vẫn train được, nhưng trạng thái không tự lưu giữa các version.')

def find_code(root):
    for _ in range(2):   # lần 2: giải nén các zip nếu Kaggle chưa giải nén
        hits = glob.glob(f'{root}/**/scripts/isbi/kaggle_runner.py', recursive=True)
        if hits:
            return os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
        for z in glob.glob(f'{root}/**/*.zip', recursive=True):
            try:
                zipfile.ZipFile(z).extractall(os.path.dirname(z))
            except OSError:   # /kaggle/input is read-only
                pass
    return None

CODE, src = '/tmp/trace_code', None
shutil.rmtree(CODE, ignore_errors=True)
if SYNC:   # bản mới nhất của dataset code (không phụ thuộc phiên bản input đang gắn)
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        dl = '/tmp/trace_code_dl'; shutil.rmtree(dl, ignore_errors=True)
        api.dataset_download_files(f"{os.environ['KAGGLE_USERNAME']}/{CODE_DATASET}", path=dl, unzip=True, quiet=True)
        src = find_code(dl)
    except Exception as e:
        print('!! không tải được dataset code qua API:', repr(e)[:300])
if src is None:
    src = find_code('/kaggle/input')
assert src, f'Không thấy code: tạo dataset {CODE_DATASET} từ TRACE-Seg3D_ISBI.zip (hoặc Add Input dataset đó)'
shutil.copytree(src, CODE)
os.chdir(CODE)
print('code từ:', src)
print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
!PYTHONPATH=.:src python -m pytest -q tests/test_trace_seg3d.py 2>&1 | tail -1

## 3. Tải trạng thái từ các version trước

In [ ]:
STATE = '/kaggle/working/trace_state'
if SYNC:
    !python scripts/isbi/kaggle_sync.py pull --state {STATE}
else:
    print('Không có KAGGLE_USERNAME/KAGGLE_KEY -> chỉ dùng trạng thái từ các input đã gắn (Notebook Output của version trước).')

## 4. Chạy đến khi hết giờ hoặc hết việc

In [ ]:
HOURS = 10.0 if SYNC else 11.0     # chừa thời gian upload trạng thái trước mốc 12 giờ của Kaggle
cmd = [sys.executable, 'scripts/isbi/kaggle_runner.py', '--input-root', '/kaggle/input', '--state', STATE, '--code', CODE,
       '--methods', PLAN['methods'], '--seeds', PLAN['seeds'], '--ablations', PLAN['ablations'],
       '--abl-source', PLAN['abl_source'], '--epochs', str(PLAN['epochs']), '--extra', PLAN['extra'],
       '--hours', str(HOURS), '--workers-per-job', '2', '--save-maps', '1']
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'Warning' not in line and 'warnings.warn' not in line:
        print(line, end='')
p.wait()
print('runner exit code', p.returncode)

## 5. Lưu trạng thái lên Kaggle Dataset

In [ ]:
meta = json.load(open(f'{STATE}/state_meta.json')) if os.path.exists(f'{STATE}/state_meta.json') else {}
note = f"version-{meta.get('version', 'x')}"
if SYNC:
    !python scripts/isbi/kaggle_sync.py push --state {STATE} --note {note}
else:
    print('Không có secrets -> trạng thái chỉ nằm trong Output của version này. Version sau: Add Input -> Notebook Output của version này.')

## 6. Kết quả
Gửi Claude file `results.zip` trong tab Output (bảng, audit, log, history; không có checkpoint).

In [ ]:
from IPython.display import Image, Markdown, display
for f in ('STATUS.md', 'results/final/table.md'):
    if os.path.exists(f'{STATE}/{f}'):
        display(Markdown(open(f'{STATE}/{f}').read()))
for png in sorted(glob.glob(f'{STATE}/check_*.png')):
    display(Image(png, width=700))
light = [p for p in glob.glob(f'{STATE}/**/*', recursive=True)
         if os.path.isfile(p) and '/processed/' not in p and p.endswith(('.json', '.csv', '.md', '.tex', '.png', '.log'))]
with zipfile.ZipFile('/kaggle/working/results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for p in light:
        z.write(p, os.path.relpath(p, '/kaggle/working'))
print(f'results.zip: {len(light)} files, {os.path.getsize("/kaggle/working/results.zip") / 1e6:.1f} MB')